# SBN Bayesian Network Sampler

Generates statistically testable datasets for Bayesian network analysis of structural
dependencies between the five binary constraints **A, R, I, H, L**.

---

## Methodological rationale

### Why constraint S is fixed to 0 in the sampling design

The module `sbn_module_v0` defines `CONSTRAINT_KEYS = ['S', 'A', 'R', 'I', 'H', 'L']`
and the full factorial space contains 64 architectures (2^6).  In this notebook,
**S is fixed to 0** and only the 32 architectures with S=0 are sampled.  This design
choice rests on two arguments:

- **No discriminative power for the BN.**  Preliminary GA runs showed that S exhibits
  no conditional dependency with the fitness outcome or with any of the remaining five
  constraints (A, R, I, H, L) across the full factorial pilot.  Including S as a free
  variable would add noise to the CPT estimates without contributing structure to the DAG.
- **Structural incomparability.**  Architectures with S=0 and S=1 differ fundamentally
  in their topology, making pooled CPT estimation unreliable.  Fixing S=0 ensures that
  all 32 sampled architectures are structurally homogeneous and comparable.

S is still present as a column in the output CSV (always 0) and in `CONSTRAINT_KEYS`,
but it is excluded from the BN variable nodes via `BN_KEYS` defined in Step 2.

### Why the existing GA result CSV files are not sufficient for a BN

The standard generator (`sbn_generator_v0.ipynb`) produces **exactly one best score
per architecture** across all 32 combinations of the 5 active constraints.  This
single-point design has two fundamental limitations for Bayesian network analysis:

1. **No statistical testability.**  Conditional independence tests (the foundation of
   both constraint-based BN learning such as the PC algorithm, and score-based learning
   such as Hill Climbing + BIC) require a *distribution* of observations per cell, not
   a single deterministic value.  With one observation per architecture, every
   conditional probability table (CPT) entry collapses to 0 or 1 — making the BN
   equivalent to a truth table, which is already what FCA provides more rigorously.

2. **Orthogonal design precludes inter-constraint inference.**  The full factorial 2^5
   design makes A...L perfectly uncorrelated by construction.  Any apparent association
   between constraint variables in the single-point CSV is an artefact of the
   experimental layout, not a property of the design space.

### Why multi-run replication is the correct solution

Running the GA **N_RUNS = 30 times** per architecture with independent random seeds
produces a genuine score distribution per architecture.  This enables:

- **CPT estimation** with proper uncertainty:
  `P(Fit_top | R=1, A=1) = 17/30 = 0.57`  rather than  `0 or 1`
- **Conditional independence tests** between constraints given score level
- **d-separation** inference to orient edges in the BN DAG

The 30-run threshold is the minimum for non-parametric tests (Mann-Whitney,
Kruskal-Wallis) to have acceptable power on binary outcomes.

### Role of the existing CSV files (reference only)

The existing CSV files placed in the directory `/data` were
produced with **gen=90** (900 evaluations/architecture) versus **gen=30** used here
(300 evaluations/architecture).  They are retained **solely as a convergence check**:
if multi-run replication scores are systematically lower than the reference CSV scores,
it indicates that 300 evaluations are insufficient for the GA to plateau, and
`NUM_GENERATIONS` should be increased before running the BN analysis.  See Step 5
(sanity check) for this verification.

### Overall pipeline

```
Multi-run replication output (32 arch x 30 runs x 3 fitness)
         |
         +--► FCA unified context (32 objects x 8 attributes)
         |        └--► concept lattice + Stem Base rules  (structure)
         |
         └--► CPT estimation from score distributions     (parameters)
                  └--► fully parameterised Bayesian Network
                           └--► inference queries
```

This notebook implements the **multi-run replication** design using the same
`sbn_module_v0` API as `sbn_generator_v0.ipynb`.


## Step 0: Dependencies

In [ ]:
# Uncomment to install if needed:
# !pip install pandas matplotlib pgmpy
print("Required: pandas, matplotlib, pgmpy")


## Step 1: Module Setup

In [ ]:
import shutil, os, glob, sys

# Adjust the source path to match your local setup
SRC = os.path.expanduser("~/Downloads/sbn_module_v0.py")
DST = os.path.expanduser("~/sbn_module_v0.py")

if os.path.exists(SRC):
    shutil.copy2(SRC, DST)
    print(f"Copied {SRC} -> {DST}")
elif os.path.exists(DST):
    print(f"Using existing {DST}")
else:
    print("WARNING: sbn_module_v0.py not found. Adjust SRC path above.")

# Clear CuPy kernel cache to avoid stale GPU code
for d in [os.path.expanduser("~/.cupy/kernel_cache"), "/tmp/cupy"]:
    if os.path.exists(d):
        for f in glob.glob(os.path.join(d, "**", "*.cubin*"), recursive=True):
            os.remove(f)

# Force module reload if already imported
for mod in list(sys.modules):
    if "sbn_module" in mod:
        del sys.modules[mod]

# Optional: CUDA library path
cuda_lib = '/opt/conda/lib/python3.13/site-packages/nvidia/cuda_nvrtc/lib'
if os.path.exists(cuda_lib):
    os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"
    print("CUDA library path configured")

print("Ready.")


## Step 2: Imports

In [ ]:
import sys, numpy as np, random, time, csv, itertools
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, FileLink

from sbn_module_v0 import (
    create_sbn, mutate_sbn, GPUAccelerator, RewardFunctions,
    run_genetic_algorithm, constraints_to_label,
    all_64_architectures, CONSTRAINT_KEYS
)

print("Imports OK")
print(f"Constraint keys: {CONSTRAINT_KEYS}")

# S is fixed to 0 in this notebook — exclude it from BN variable nodes
BN_KEYS = [k for k in CONSTRAINT_KEYS if k != 'S']
print(f"BN variable keys (S=0 fixed): {BN_KEYS}")


## Step 3: GPU & Fitness Setup

In [ ]:
gpu     = GPUAccelerator()
rewards = RewardFunctions(gpu)
print("GPU ready" if gpu.available else "CPU mode")


## Step 3b — Layer 2/3 Metric Extraction Functions

These functions are applied to the **best SBN** returned by each GA run,
adding two new layers to the output dataset:

**Layer 2 — Topological properties of the circuit graph**
- `nl_fraction` : fraction of non-linear gates (AND, OR) among all logic gates
- `dep_density` : fraction of 1s in the 16x16 dependency matrix D[i,j]=1 if
  output bit i depends on input bit j
- `avg_depth`   : average topological depth of the 16 output nodes

**Layer 3 — Shannon mechanisms**
- `confusion`  : normalised linear resistance = how far the function is from
  any affine function (Shannon confusion)
- `diffusion`  : average avalanche coefficient = fraction of output bits that
  change when one input bit is flipped, averaged over 256 random inputs and
  all 16 input positions (SAC approximation, perfect diffusion = 0.5)

All five metrics are computed **in addition to** `best_score` and recorded
in the output CSV.  The fitness being optimised does not affect these metrics.


In [ ]:
# ---------------------------------------------------------------------------
# Layer 2: topological properties
# ---------------------------------------------------------------------------

def nl_fraction(sbn):
    """Fraction of non-linear gates (AND, OR) among all logic gates."""
    ops = sbn.analyze_structure()['operations']
    nl    = ops.get('AND', 0) + ops.get('OR', 0)
    total = sum(v for k, v in ops.items() if k != 'IDENTITY')
    return float(nl / total) if total > 0 else 0.0


def dep_density(sbn):
    """Fraction of 1s in the 16x16 dependency matrix.
    D[i,j] = 1 iff output bit i depends on input bit j.
    """
    D = np.zeros((16, 16), dtype=np.int8)
    for bit in range(16):
        output_id = sbn.output_map[bit]
        visited   = set()
        stack     = [output_id]
        while stack:
            nid = stack.pop()
            if nid in visited:
                continue
            visited.add(nid)
            if nid.startswith('input_'):
                try:
                    D[bit, int(nid.split('_')[1])] = 1
                except ValueError:
                    pass
            elif nid in sbn.nodes:
                stack.extend(sbn.nodes[nid].inputs)
    return float(np.sum(D)) / 256.0


def avg_depth(sbn):
    """Average topological depth of the 16 output nodes.
    Depth = length of the longest path from any input to this node.
    Cycle-safe: nodes on cycles receive depth 0.
    """
    cache    = {}
    in_stack = set()

    def depth(nid):
        if nid in cache:
            return cache[nid]
        if nid.startswith('input_') or nid.startswith('state_'):
            cache[nid] = 0
            return 0
        if nid not in sbn.nodes or nid in in_stack:
            cache[nid] = 0
            return 0
        in_stack.add(nid)
        d = 1 + max((depth(inp) for inp in sbn.nodes[nid].inputs), default=0)
        in_stack.discard(nid)
        cache[nid] = d
        return d

    depths = [depth(sbn.output_map[b]) for b in range(16)]
    return float(np.mean(depths))


# ---------------------------------------------------------------------------
# Layer 3: Shannon mechanisms
# ---------------------------------------------------------------------------

def confusion_score(sbn, gpu_acc):
    """Normalised linear resistance in [0, 1].
    1 = maximally non-linear (perfect confusion).
    """
    return float(gpu_acc.linear_resistance(sbn)) / 32640.0


def diffusion_score(sbn, n_samples=256):
    """Average avalanche coefficient (SAC approximation).
    For n_samples random inputs and all 16 input positions, measure
    the fraction of output bits that change when one input bit is flipped.
    Perfect diffusion = 0.5.
    """
    total = 0.0
    count = 0
    for _ in range(n_samples):
        x = [random.randint(0, 1) for _ in range(16)]
        sbn.reset([0] * 16)
        out_x = sbn.step(x[:])
        for i in range(16):
            x_flip    = x[:]
            x_flip[i] ^= 1
            sbn.reset([0] * 16)
            out_flip  = sbn.step(x_flip)
            total    += sum(a != b for a, b in zip(out_x, out_flip)) / 16.0
            count    += 1
    return float(total / count) if count > 0 else 0.0


def extract_all_metrics(best_sbn, gpu_acc):
    """Return all Layer 2 and Layer 3 metrics for a given SBN."""
    return {
        'nl_fraction': round(nl_fraction(best_sbn),  4),
        'dep_density': round(dep_density(best_sbn),  4),
        'avg_depth':   round(avg_depth(best_sbn),    4),
        'confusion':   round(confusion_score(best_sbn, gpu_acc), 4),
        'diffusion':   round(diffusion_score(best_sbn), 4),
    }


# Quick smoke test
from sbn_module_v0 import create_sbn
_test_sbn = create_sbn(S=True, R=True)
_m = extract_all_metrics(_test_sbn, gpu)
print('Smoke test metrics:', _m)
print('All Layer 2/3 functions ready.')


## Step 4: Global Configuration

Parameters for the multi-run replication.


In [ ]:
# ---------------------------------------------------------------------------
# Fitness function
# ---------------------------------------------------------------------------
# Choose one: 'algebraic', 'linear', 'differential'
SELECTED_FITNESS = "algebraic"

# Methods live on the gpu object (mirrors sbn_generator_v0 convention):
#   gpu.algebraic_degree        — algebraic degree via ANF  (maximize)
#   gpu.linear_resistance       — linear complexity via WHT (maximize)
#   gpu.differential_resistance — differential resistance   (minimize)
FITNESS_MAP = {
    "algebraic":    (gpu.algebraic_degree,        True),
    "linear":       (gpu.linear_resistance,       True),
    "differential": (gpu.differential_resistance, False),
}
fitness_fn, maximize = FITNESS_MAP[SELECTED_FITNESS]
print(f"Fitness: {SELECTED_FITNESS}  |  maximize={maximize}")

# ---------------------------------------------------------------------------
# Shared GA hyper-parameters (kept low for quick iteration; increase for
# production runs)
# ---------------------------------------------------------------------------
POPULATION_SIZE  = 10    # individuals per generation
NUM_GENERATIONS  = 30    # generations per run
MUTATION_RATE    = 2.0   # Poisson mean for mutations per individual

print(f"Population={POPULATION_SIZE}  Generations={NUM_GENERATIONS}  Mutation={MUTATION_RATE}")


---
## Multi-Run Replication

Each of the 32 architectures is run **N_RUNS** times with independent random seeds.
This produces a *distribution* of best scores per architecture rather than a single
point, enabling:
- Mann-Whitney U tests per constraint (S=0 vs S=1, etc.)
- Two-way ANOVA for pairwise interactions (e.g. R×A)
- BN score-based structure learning with `pgmpy` (discretised score as target node)

### Output schema
```
run_id | arch_id | S | A | R | I | H | L | seed | best_score | time_s
```


In [ ]:
# ---------------------------------------------------------------------------
# Multi-run replication configuration
# ---------------------------------------------------------------------------
# IMPORTANT: differential_resistance costs ~12s/eval (GPU DDT over 65535 inputs).
# For differential, GA optimisation is replaced by direct random sampling:
# pop=1, gen=1 draws one random SBN per architecture per run.
# This is statistically valid for BN CPT estimation because the architectural
# constraints (A,R,I,H,L) influence the score distribution even without
# optimisation. The tradeoff is higher variance in CPT entries for differential
# compared to algebraic/linear, which is acceptable and documented in Step 5.
#
#  Fitness        | pop | gen | time/eval | total (30 runs x 32 arch)
#  ---------------|-----|-----|-----------|---------------------------
#  algebraic      |  10 |  30 |   0.01s   |  ~2h15
#  linear         |  10 |  30 |   0.01s   |  ~2h15
#  differential   |   1 |   1 |   12s     |  ~3h12  ← random sampling
# ---------------------------------------------------------------------------

N_RUNS    = 30     # independent replications per architecture (never reduce)
BASE_SEED = 1000   # seeds = BASE_SEED + run_idx * 32 + arch_idx

# Override GA parameters for differential
if SELECTED_FITNESS == "differential":
    POPULATION_SIZE = 1   # no optimisation — pure random sampling
    NUM_GENERATIONS = 1   # one evaluation per run per architecture
    print("differential mode: random sampling (pop=1, gen=1)")
    print("Note: scores reflect raw architectural distribution, not GA optima.")
    print("CPT variance will be higher than for algebraic/linear — expected.")

time_per_eval = {"algebraic": 0.01, "linear": 0.01, "differential": 12.0}
est_hours = N_RUNS * 32 * POPULATION_SIZE * NUM_GENERATIONS \
            * time_per_eval[SELECTED_FITNESS] / 3600
print(f"Total rows in output CSV : {N_RUNS * 32}")
print(f"Estimated runtime        : {est_hours:.1f} hours")


In [ ]:
# ---------------------------------------------------------------------------
# Multi-run replication execution — records Layer 2 and Layer 3 metrics
# ---------------------------------------------------------------------------
RUN = False  # <-- set to True to run

if not RUN:
    print('Sampler skipped (RUN=False). Set RUN=True to execute.')
else:
    architectures = [a for a in all_64_architectures() if not a['S']]
    assert len(architectures) == 32, f"Expected 32 S=0 architectures, got {len(architectures)}"
    records    = []
    t_start       = time.time()

    for run_idx in range(N_RUNS):
        for arch_idx, arch in enumerate(architectures):
            seed = BASE_SEED + run_idx * 32 + arch_idx
            np.random.seed(seed)
            random.seed(seed)

            label = constraints_to_label(arch)
            t0    = time.time()

            # GA returns the best SBN object — used for Layer 2/3 extraction
            best_sbn, best_score, _ = run_genetic_algorithm(
                constraints=arch,
                fitness_fn=fitness_fn,
                maximize=maximize,
                population_size=POPULATION_SIZE,
                num_generations=NUM_GENERATIONS,
                mutation_rate=MUTATION_RATE,
            )

            elapsed = time.time() - t0

            # Extract Layer 2 and Layer 3 metrics from the best SBN
            metrics = extract_all_metrics(best_sbn, gpu)

            records.append({
                'run_id':       run_idx,
                'arch_id':      arch_idx,
                'Architecture': label,
                **{k: int(arch[k]) for k in CONSTRAINT_KEYS},
                'seed':         seed,
                'best_score':   round(float(best_score), 6),
                **metrics,
                'time_s':       round(elapsed, 2),
            })

        elapsed_total = time.time() - t_start
        eta = elapsed_total / (run_idx + 1) * (N_RUNS - run_idx - 1)
        print(f'Run {run_idx+1}/{N_RUNS} done — '
              f'elapsed {elapsed_total/60:.1f} min, ETA {eta/60:.1f} min')

    df_out = pd.DataFrame(records)
    fname = (f'multirep_{SELECTED_FITNESS}'
             f'_pop{POPULATION_SIZE}_gen{NUM_GENERATIONS}_n{N_RUNS}.csv')
    df_out.to_csv(fname, index=False)
    display(FileLink(fname, result_html_prefix='Download CSV: '))
    print(f'Saved {len(df_out)} rows -> {fname}')
    print(f'Columns: {list(df_out.columns)}')
    display(df_out.head(5))


---
## Step 5: Convergence Check and Sanity Verification

### Purpose

Before using the multi-run replication data for BN analysis, two checks are mandatory:

**Check A — Convergence adequacy.**  
The multi-run replication uses `NUM_GENERATIONS=30` (300 evaluations/architecture)
while the reference CSV files used `gen=90` (900 evaluations/architecture).  If the
GA has not plateaued by generation 30, scores will be systematically underestimated
and the CPT entries will reflect optimisation noise rather than architectural
properties.  
**Decision rule**: if median score < 90% of reference score for more than 20% of
architectures, increase `NUM_GENERATIONS` and rerun.

**Check B — Distribution quality.**  
Each architecture should show meaningful variance across 30 runs.  Architectures
with zero variance (GA always finds the same score) are valid but will contribute
deterministic CPT entries — this is acceptable and expected for highly constrained
architectures.


In [ ]:
# ---------------------------------------------------------------------------
# Load files
# ---------------------------------------------------------------------------
import glob as _glob

multirep_files = sorted(_glob.glob("e1_multirep_*.csv"))
ref_files = sorted(_glob.glob("/data/ga_results_*.csv"))   # reference CSVs (gen=90)

print("Multi-run replication files :", multirep_files)
print("Reference files             :", ref_files)


In [ ]:
# ---------------------------------------------------------------------------
# Check A — Convergence: multi-run median vs reference best_score per architecture
# ---------------------------------------------------------------------------
# Requires one multi-run replication file and the matching reference CSV for
# the same fitness.  Edit the two paths below to match your files.
# ---------------------------------------------------------------------------
MULTIREP_FILE = multirep_files[-1] if multirep_files else None
REF_FILE = None          # e.g. "/data/ga_results_algebraic_pop10_gen90.csv"

# Auto-match reference file by fitness name
if MULTIREP_FILE and not REF_FILE:
    fitness_tag = MULTIREP_FILE.split('_')[2]          # e.g. 'algebraic'
    matches = [f for f in ref_files if fitness_tag in f]
    REF_FILE = matches[0] if matches else None

if not MULTIREP_FILE:
    print("No multi-run replication file found — run the sampler first.")
elif not REF_FILE:
    print(f"No reference CSV found for fitness '{fitness_tag}'.")
    print("Place ga_results_*.csv files in /data.")
else:
    df_mr  = pd.read_csv(MULTIREP_FILE)
    df_ref = pd.read_csv(REF_FILE)

    # Multi-run: median score per architecture
    mr_med = df_mr.groupby("arch_id")["best_score"].median().reset_index()
    mr_med.columns = ["arch_id", "mr_median"]

    # Reference: single best score per architecture (sorted by Rank)
    # arch_id is inferred from row order (0-31) matching all_64_architectures()
    df_ref = df_ref.reset_index(drop=True)
    df_ref["arch_id"] = df_ref.index
    ref_best = df_ref[["arch_id", "Best_Score"]].rename(
        columns={"Best_Score": "ref_best"})

    cmp = mr_med.merge(ref_best, on="arch_id")

    # For differential (minimize), lower is better — invert ratio
    if not maximize:
        cmp["ratio"] = cmp["ref_best"] / cmp["mr_median"].replace(0, np.nan)
    else:
        cmp["ratio"] = cmp["mr_median"] / cmp["ref_best"].replace(0, np.nan)

    below_threshold = (cmp["ratio"] < 0.90).sum()
    pct_below = below_threshold / len(cmp) * 100

    print("=" * 60)
    print("CHECK A — Convergence adequacy (multi-run gen=30 vs ref gen=90)")
    print("=" * 60)
    print(f"Architectures with multi-run median < 90% of reference : "
          f"{below_threshold}/{len(cmp)}  ({pct_below:.1f}%)")

    if pct_below > 20:
        print()
        print("WARNING: convergence insufficient.")
        print("Recommendation: increase NUM_GENERATIONS and rerun.")
    else:
        print("OK: scores plateau within 10% of reference.")
        print("300 evaluations are sufficient for this fitness.")

    print()
    print("Ratio distribution (multi-run median / ref best):")
    print(cmp["ratio"].describe().round(3))

    # Plot
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(cmp["arch_id"], cmp["ratio"], color=[
        "#e74c3c" if r < 0.90 else "#2ecc71" for r in cmp["ratio"]
    ], alpha=0.8)
    ax.axhline(0.90, color="#e74c3c", linestyle="--", linewidth=1.5,
               label="90% threshold")
    ax.axhline(1.00, color="#555",    linestyle=":",  linewidth=1.0)
    ax.set_xlabel("Architecture index", fontsize=11)
    ax.set_ylabel("Multi-run median / ref best", fontsize=11)
    ax.set_title("Check A — Convergence: multi-run (gen=30) vs reference (gen=90)",
                 fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Check B — Distribution quality across 30 runs
# ---------------------------------------------------------------------------
if multirep_files:
    df_mr = pd.read_csv(multirep_files[-1])

    stats = df_mr.groupby("arch_id")["best_score"].agg(
        mean="mean", std="std", cv=lambda x: x.std()/x.mean() if x.mean() != 0 else 0
    ).reset_index()

    zero_var = (stats["std"] == 0).sum()
    n_arch = len(stats)
    print("=" * 60)
    print("CHECK B — Score distribution quality across 30 runs")
    print("=" * 60)
    print(f"Architectures with zero variance : {zero_var}/{n_arch}")
    print(f"  (deterministic CPT entries — acceptable)")
    print()
    print("CV distribution (coefficient of variation):")
    print(stats["cv"].describe().round(3))
    print()
    print("Score statistics per constraint (mean of arch medians):")
    for k in BN_KEYS:
        g = df_mr.groupby(k)["best_score"].agg(["mean", "std"])
        print(f"  {k}:  0 → {g.loc[0,'mean']:.3f} ± {g.loc[0,'std']:.3f}  "
              f"1 → {g.loc[1,'mean']:.3f} ± {g.loc[1,'std']:.3f}")
else:
    print("No multi-run replication file available yet.")
